# Priority Classifier v2

**Nessun prerequisito Colab** — gli embedding vengono calcolati localmente su CPU (o caricati dalla cache se già esistenti).

### Pipeline
1. Carica `dataset_clean.csv` e split temporale
2. Calcola embedding **intfloat/multilingual-e5-base** su CPU (con caching su disco)
3. Feature engineering (OHE + has_urgenza)
4. Tuning C con 3-fold CV
5. Training LinearSVC + valutazione
6. Salvataggio in `modelli/priority_v2/`


---
## STEP 1 — Caricamento dataset e split temporale

In [ ]:
import pandas as pd
import numpy as np
import warnings
import time
import os
from pathlib import Path
warnings.filterwarnings('ignore')

# Il kernel VS Code parte dalla cartella notebooks/, non dalla root del progetto.
# os.path.abspath('') restituisce la directory corrente del kernel,
# quindi .parent risale di un livello e otteniamo la root di TicketClassifier/.
BASE_DIR = Path(os.path.abspath('')).parent

DATA_DIR = BASE_DIR / 'data'
EMB_DIR  = BASE_DIR / 'embeddings'
MOD_DIR  = BASE_DIR / 'modelli'

MODEL_NAME     = 'intfloat/multilingual-e5-base'
EMB_TRAIN_PATH = EMB_DIR / 'priority_e5_train.npy'
EMB_TEST_PATH  = EMB_DIR / 'priority_e5_test.npy'

# Lo split e' temporale (non random) per simulare un vero scenario produttivo:
# il modello viene addestrato su dati storici e valutato su dati futuri.
# Uno split random sovrastimerebbe le performance perche' ticket simili
# (stesso cliente, stesso tipo di problema) finirebbero sia in train che in test.
SOGLIA_SPLIT = '2025-11-01'

df = pd.read_csv(DATA_DIR / 'dataset_clean.csv', parse_dates=['data_creazione'])

df_train = df[df['data_creazione'] < SOGLIA_SPLIT].copy()
df_test  = df[df['data_creazione'] >= SOGLIA_SPLIT].copy()

print(f"Dataset totale  : {len(df):,} ticket")
print(f"Train (<{SOGLIA_SPLIT})  : {len(df_train):,}")
print(f"Test  (>={SOGLIA_SPLIT}) : {len(df_test):,}")
print(f"\nDistribuzione priorita' — train:")
print(df_train['priorita_finale'].value_counts().sort_index())
print(f"\nDistribuzione priorita' — test:")
print(df_test['priorita_finale'].value_counts().sort_index())

---
## STEP 2 — Calcolo embedding E5 su CPU (con caching su disco)

Il testo viene prefissato con `query: ` come richiesto dal modello **intfloat/multilingual-e5-base**.  
Se i file `.npy` esistono gia' nella cartella `embeddings/` vengono caricati direttamente,
altrimenti vengono calcolati su CPU e salvati per le run successive.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME     = 'intfloat/multilingual-e5-base'
EMB_TRAIN_PATH = EMB_DIR / 'priority_e5_train.npy'
EMB_TEST_PATH  = EMB_DIR / 'priority_e5_test.npy'


def prepara_testo_e5(serie: pd.Series) -> list:
    """
    Aggiunge il prefisso 'query: ' richiesto dal modello multilingual-e5.
    Senza questo prefisso le rappresentazioni vettoriali risultano peggiori
    perche' il modello e' stato addestrato distinguendo 'query' da 'passage'.
    """
    return ['query: ' + str(t) for t in serie.fillna('')]


def calcola_o_carica_embedding(df_split: pd.DataFrame, path_npy: Path) -> np.ndarray:
    """
    Carica gli embedding da disco se esistono, altrimenti li calcola su CPU e li salva.
    Il caching e' fondamentale: il calcolo completo su CPU richiede circa 30-40 minuti
    per 46k testi, quindi e' impensabile ricalcolarlo ad ogni run.
    """
    if path_npy.exists():
        print(f"  Carico embedding dal file: {path_npy.name}")
        emb = np.load(path_npy)
        # Verifico che il file su disco corrisponda al dataset corrente
        assert len(emb) == len(df_split), (
            f"Dimensioni non corrispondono: {len(emb)} embedding vs {len(df_split)} righe. "
            f"Cancella {path_npy.name} e ricalcola."
        )
        return emb

    print(f"  Calcolo embedding con '{MODEL_NAME}' su CPU...")
    print(f"  Testi da codificare: {len(df_split):,}")

    # device='cpu' per compatibilita' con qualsiasi macchina, senza bisogno di CUDA
    modello = SentenceTransformer(MODEL_NAME, device='cpu')
    testi = prepara_testo_e5(df_split['testo_input'])

    t0 = time.time()
    emb = modello.encode(
        testi,
        batch_size=64,             # batch piu' grandi accelerano ma consumano piu' RAM
        normalize_embeddings=True, # L2-normalizza ogni vettore: con norma=1 il dot product e' uguale alla cosine similarity
        show_progress_bar=True,
    )
    print(f"  Completato in {time.time() - t0:.0f}s")
    np.save(path_npy, emb)
    print(f"  Salvato in: {path_npy}")
    return emb


print(f"Modello di embedding: {MODEL_NAME}")
print()
print("Train:")
X_emb_train = calcola_o_carica_embedding(df_train, EMB_TRAIN_PATH)
print("Test:")
X_emb_test  = calcola_o_carica_embedding(df_test, EMB_TEST_PATH)

print(f"\nShape embedding train : {X_emb_train.shape}")
print(f"Shape embedding test  : {X_emb_test.shape}")

# Con normalize_embeddings=True ogni norma deve essere ~1.0
norme = np.linalg.norm(X_emb_train[:5], axis=1)
print(f"Norma L2 primi 5 vettori (atteso ~1.0): {norme}")

---
## STEP 3 — Feature engineering strutturali

Affianchiamo agli embedding due feature strutturali gia' disponibili al momento della predizione:

**`priorita_iniziale_cliente`** (One-Hot Encoded):  
Il cliente sceglie la priorita' nel form ZConnect — disponibile in produzione prima ancora di leggere il testo.  
L'analisi dei pesi (v1) mostra che questa feature pesa ~2x rispetto agli embedding.
Questo e' corretto e non e' data leakage: l'88% delle volte l'operatore non la corregge.

**`has_urgenza`** (booleana):  
1 se il testo contiene: 'urgente', 'immediato', 'il prima possibile', 'critico'.  
43.5% dei ticket con urgenza sono P1 vs 12.2% senza — segnale forte.

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import OneHotEncoder

CAT_COLS  = ['priorita_iniziale_cliente']
BOOL_COLS = ['has_urgenza']

# Gestione NaN: la priorita' cliente e' valorizzata solo nel ~12% dei ticket,
# il resto viene trattato come categoria 'sconosciuto' (non scartato)
df_train[CAT_COLS] = df_train[CAT_COLS].fillna('sconosciuto')
df_test[CAT_COLS]  = df_test[CAT_COLS].fillna('sconosciuto')

# Il fit viene fatto solo su train per evitare data leakage:
# il modello non deve "vedere" quali categorie esistono nel test set.
# handle_unknown='ignore': se nel test compare una categoria non vista in train,
# produce un vettore di zeri invece di dare errore
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(df_train[CAT_COLS])

X_cat_train = encoder.transform(df_train[CAT_COLS])
X_cat_test  = encoder.transform(df_test[CAT_COLS])

# has_urgenza e' gia' 0/1, basta convertirla in matrice sparsa per poterla
# concatenare con gli altri blocchi tramite hstack
X_bool_train = sp.csr_matrix(df_train[BOOL_COLS].values.astype(float))
X_bool_test  = sp.csr_matrix(df_test[BOOL_COLS].values.astype(float))

print(f"Feature categoriche (OHE): {X_cat_train.shape[1]} colonne")
print(f"  Categorie: {encoder.categories_[0].tolist()}")
print(f"Feature booleane:          {X_bool_train.shape[1]} colonna (has_urgenza)")

# Sanity check: verifico che la correlazione urgenza→priorita' sia quella attesa
# (da EDA: 43.5% dei ticket con urgenza sono P1 vs 12.2% senza)
print(f"\nCorrelazione has_urgenza → priorita' (train):")
urgenza_arr  = df_train['has_urgenza'].values
priority_arr = df_train['priorita_finale'].values
for p in ['P1', 'P2', 'P3', 'P4']:
    pct = urgenza_arr[priority_arr == p].mean() * 100
    print(f"  {p}: {pct:.1f}% dei ticket ha has_urgenza=1")

---
## STEP 4 — Costruzione feature matrix finale

Combiniamo le tre componenti in un'unica matrice sparsa:

```
X = [  embedding E5 (768d)  |  OHE priorita_iniziale (4d)  |  has_urgenza (1d)  ]
      └── segnale testuale ──┘  └──── segnale cliente ─────┘  └── segnale urgenza ┘
```

Totale: **773 feature** per ticket.

In [ ]:
# Converti embedding densi in sparse per hstack
X_emb_train_sp = sp.csr_matrix(X_emb_train)
X_emb_test_sp  = sp.csr_matrix(X_emb_test)

# hstack: unisce colonna per colonna i tre blocchi di feature
X_train = sp.hstack([X_emb_train_sp, X_cat_train, X_bool_train])
X_test  = sp.hstack([X_emb_test_sp,  X_cat_test,  X_bool_test])

y_train = df_train['priorita_finale'].values
y_test  = df_test['priorita_finale'].values

print(f"Feature matrix train: {X_train.shape}")
print(f"Feature matrix test:  {X_test.shape}")
print(f"  └── embedding:   768 colonne")
print(f"  └── OHE categ.:  {X_cat_train.shape[1]} colonne")
print(f"  └── has_urgenza: 1 colonna")
print(f"  └── TOTALE:      {X_train.shape[1]} colonne")
print(f"\nClassi target: {np.unique(y_train)}")

---
## STEP 5 — Tuning parametro C con cross-validation

Cerchiamo il C ottimale per LinearSVC.  
C regola il tradeoff bias/varianza:
- **C piccolo** (0.01): forte regolarizzazione, puo' underfitare
- **C grande** (10): poca regolarizzazione, puo' overfitare su train

Con embedding gia' L2-normalizzati, i valori ottimali tipici sono tra 0.1 e 5.0.  
Usiamo 3-fold CV (non 5) per velocita' — su 46k ticket e' gia' robusto.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score, StratifiedKFold

# C regola il tradeoff bias/varianza in LinearSVC:
# valori piccoli (0.01) → forte regolarizzazione, puo' underfitare
# valori grandi (10)    → poca regolarizzazione, puo' overfitare su train
# Con embedding gia' L2-normalizzati i valori ottimali tipici stanno tra 0.1 e 5.0
VALORI_C = [0.1, 0.5, 1.0, 5.0]

# StratifiedKFold garantisce che ogni fold abbia la stessa proporzione di classi,
# importante perche' P4 e' solo il 2.5% del dataset e uno split casuale potrebbe
# escluderlo quasi del tutto da qualche fold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Tuning parametro C — LinearSVC (3-fold CV, metrica: macro F1)")
print(f"{'C':>6}  {'F1 macro':>10}  {'Std':>8}  {'Tempo':>8}")
print("-" * 42)

risultati_c = {}
for c in VALORI_C:
    clf_tmp = LinearSVC(class_weight='balanced', max_iter=3000, C=c, random_state=42)
    t0 = time.time()
    scores = cross_val_score(
        clf_tmp, X_train, y_train,
        cv=cv, scoring='f1_macro', n_jobs=-1
    )
    elapsed = time.time() - t0
    risultati_c[c] = scores.mean()
    print(f"{c:>6.2f}  {scores.mean():>10.4f}  {scores.std():>8.4f}  {elapsed:>7.0f}s")

C_OTTIMALE = max(risultati_c, key=risultati_c.get)
print(f"\nC ottimale selezionato: {C_OTTIMALE}  (F1 macro = {risultati_c[C_OTTIMALE]:.4f})")

---
## STEP 6 — Training e valutazione finale

In [ ]:
from sklearn.metrics import classification_report, f1_score

print(f"Training LinearSVC con C={C_OTTIMALE}...")
t0 = time.time()

clf = LinearSVC(
    class_weight='balanced',  # compensa lo sbilanciamento: senza questo P4 (2.5%) verrebbe quasi ignorato
    max_iter=3000,
    C=C_OTTIMALE,
    random_state=42
)
clf.fit(X_train, y_train)
print(f"Training completato in {time.time()-t0:.1f}s")

y_pred = clf.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy  = (y_pred == y_test).mean()

print(f"\n{'='*60}")
print(f"RISULTATI CLASSIFICATORE PRIORITA' v2 (E5-base + LinearSVC)")
print(f"{'='*60}")
print(f"Macro F1:  {macro_f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print()
print(classification_report(y_test, y_pred, digits=3))

---
## STEP 7 — Confronto v1 vs v2 e analisi pesi

Confronto diretto con i risultati della v1 (mpnet + split temporale Nov 2025).

In [ ]:
# Risultati v1 per confronto (mpnet, split temporale, C=1.0 non tunato)
v1_results = {
    'P1': {'f1': 0.760},
    'P2': {'f1': 0.840},
    'P3': {'f1': 0.950},
    'P4': {'f1': 0.840},
    'macro_f1': 0.843,
    'accuracy': 0.880
}

from sklearn.metrics import precision_recall_fscore_support

macro_v2 = f1_score(y_test, y_pred, average='macro')
acc_v2   = (y_pred == y_test).mean()

print("=" * 65)
print(f"{'Metrica':<30} {'v1 (mpnet)':>12} {'v2 (E5)':>12} {'Delta':>8}")
print("-" * 65)
print(f"{'Macro F1':<30} {v1_results['macro_f1']:>12.3f} {macro_v2:>12.3f} {macro_v2-v1_results['macro_f1']:>+8.3f}")
print(f"{'Accuracy':<30} {v1_results['accuracy']:>12.3f} {acc_v2:>12.3f} {acc_v2-v1_results['accuracy']:>+8.3f}")

# f1_score e' gia' importato dallo step 5, qui uso precision_recall_fscore_support
# per ottenere i valori per classe in un'unica chiamata
_, _, f1_per_class, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=['P1', 'P2', 'P3', 'P4']
)
for i, classe in enumerate(['P1', 'P2', 'P3', 'P4']):
    f1_v1 = v1_results[classe]['f1']
    print(f"{'F1 '+classe:<30} {f1_v1:>12.3f} {f1_per_class[i]:>12.3f} {f1_per_class[i]-f1_v1:>+8.3f}")
print("=" * 65)

In [ ]:
# Analisi importanza gruppi di feature (embedding vs categoriche vs urgenza)
# Stesso calcolo della v1 — confrontiamo i valori

n_emb  = 768
n_cat  = X_cat_train.shape[1]
n_bool = 1  # has_urgenza

print("=== Peso medio assoluto per gruppo di feature ===")
print(f"{'Classe':<8} {'Embedding':>12} {'Priorità_cli':>14} {'has_urgenza':>13}")
print("-" * 52)

for i, classe in enumerate(clf.classes_):
    coef = clf.coef_[i]
    peso_emb  = np.abs(coef[:n_emb]).mean()
    peso_cat  = np.abs(coef[n_emb:n_emb+n_cat]).mean()
    peso_bool = np.abs(coef[n_emb+n_cat:]).mean()
    print(f"{classe:<8} {peso_emb:>12.4f} {peso_cat:>14.4f} {peso_bool:>13.4f}")

print()
print("Nota: se 'Embedding' < 'Priorità_cli' significa che il modello")
print("si fida più della scelta del cliente che del testo del ticket.")
print("Questo è atteso (88% dei casi non viene corretto dall'operatore).")

---
## STEP 8 — Confusion matrix e analisi errori

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix assoluta
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=['P1', 'P2', 'P3', 'P4'],
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix — Assoluta')

# Confusion matrix normalizzata per riga (recall per classe)
cm_norm = confusion_matrix(y_test, y_pred, labels=['P1','P2','P3','P4'], normalize='true')
import seaborn as sns
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=['P1','P2','P3','P4'], yticklabels=['P1','P2','P3','P4'],
    ax=axes[1], vmin=0, vmax=1
)
axes[1].set_title('Confusion Matrix — Normalizzata (recall)')
axes[1].set_xlabel('Predizione'); axes[1].set_ylabel('Verità')

plt.suptitle('Priority Classifier v2 — E5-base + LinearSVC', fontsize=13)
plt.tight_layout()
plt.show()

# Errori più frequenti
df_err = pd.DataFrame({'reale': y_test, 'predetto': y_pred})
errori = df_err[df_err['reale'] != df_err['predetto']]
print(f"\nErrori totali: {len(errori):,} su {len(y_test):,} ({len(errori)/len(y_test)*100:.1f}%)")
print("\nTop coppie di confusione:")
print(
    errori.groupby(['reale', 'predetto']).size()
    .sort_values(ascending=False).head(8).to_string()
)

---
## STEP 9 — Salvataggio modello

In [ ]:
import joblib
import json

SAVE_DIR = MOD_DIR / 'priority_v2'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Salvo classificatore e OHE separatamente:
# in produzione servono entrambi — prima si OHE la priorita' cliente,
# poi si costruisce X e si predice con il classificatore
joblib.dump(clf,     SAVE_DIR / 'classificatore_svc.pkl')
joblib.dump(encoder, SAVE_DIR / 'ohe_encoder.pkl')

# Il file metadata.json contiene tutto il necessario per ricaricare
# e usare il modello in un altro script senza riaprire il notebook
metadata = {
    'versione': 'v2',
    'modello_embedding': 'intfloat/multilingual-e5-base',
    'prefisso_e5': 'query: ',           # da anteporre a ogni testo prima di chiamare .encode()
    'classificatore': 'LinearSVC',
    'C_ottimale': C_OTTIMALE,
    'feature': [
        'embedding_e5_768d',
        'priorita_iniziale_cliente_ohe',
        'has_urgenza'
    ],
    'classi': clf.classes_.tolist(),
    'split': 'temporale',
    'soglia_split': SOGLIA_SPLIT,
    'macro_f1_test': round(float(f1_score(y_test, y_pred, average='macro')), 4),
    'accuracy_test': round(float((y_pred == y_test).mean()), 4),
    'n_train': int(len(y_train)),
    'n_test': int(len(y_test)),
    'keyword_urgenza': ['urgente', 'immediato', 'il prima possibile', 'critico']
}

with open(SAVE_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Modello salvato in: {SAVE_DIR}")
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))